# 🤖 AI FAQ Chatbot

## CodeAlpha Artificial Intelligence Internship

### Project Overview

This project implements an intelligent FAQ Chatbot using Natural Language Processing (NLP).

The chatbot retrieves the most relevant answer for a user's question by applying:

- Text Preprocessing
- TF-IDF Vectorization
- Cosine Similarity

The project uses the Twitter Customer Support dataset and provides real-time responses through a Streamlit web application.

---

### Technologies Used

- Python
- Pandas
- NumPy
- NLTK
- Scikit-learn
- Streamlit
- Joblib

---

### Machine Learning Pipeline

Dataset

↓

Data Cleaning

↓

Text Preprocessing

↓

TF-IDF Vectorization

↓

Cosine Similarity

↓

Best Matching Question

↓

Display Answer

**Import Libraries**

In [21]:
# ============================================================
# Section 1: Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import re
import string
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

print("Libraries Imported Successfully")

Libraries Imported Successfully


**Upload Datasets**

In [22]:
# ============================================================
# Section 2: Load Dataset
# ============================================================

df = pd.read_csv("/content/Customer Support Data Set - Sheet1 (1).csv")

print("Dataset Loaded Successfully")
print(df.shape)

df.head()

Dataset Loaded Successfully
(500, 3)


,question,answer,category
0,Where is my order?,You can track your order using the tracking li...,Shipping
1,How do I return a shirt that doesn't fit?,You can return any item within 30 days in its ...,Returns
2,"My package hasn't arrived yet, what should I do?",Please check the tracking number. If it hasn't...,Shipping
3,Can I exchange a dress for a different size?,"Yes, use our exchange form online to select th...",Returns
4,Do you offer free shipping?,"Yes, orders over $50 qualify for free standard...",Shipping


In [23]:
# ============================================================
# Section 3: Dataset Information
# ============================================================

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  500 non-null    object
 1   answer    500 non-null    object
 2   category  500 non-null    object
dtypes: object(3)
memory usage: 11.8+ KB


In [24]:
# ============================================================
# Section 4: Missing Values
# ============================================================

print(df.isnull().sum())

question    0
answer      0
category    0
dtype: int64


In [25]:
# ============================================================
# Section 5: Duplicate Records
# ============================================================

print("Duplicate Records :", df.duplicated().sum())

Duplicate Records : 0


In [26]:
# ============================================================
# Section 6: Column Names
# ============================================================

print(df.columns)

Index(['question', 'answer', 'category'], dtype='object')


In [27]:
# ============================================================
# Section 7: Dataset Preview
# ============================================================

print("="*60)
print("First Five Records")
print("="*60)

df.head()

First Five Records


,question,answer,category
0,Where is my order?,You can track your order using the tracking li...,Shipping
1,How do I return a shirt that doesn't fit?,You can return any item within 30 days in its ...,Returns
2,"My package hasn't arrived yet, what should I do?",Please check the tracking number. If it hasn't...,Shipping
3,Can I exchange a dress for a different size?,"Yes, use our exchange form online to select th...",Returns
4,Do you offer free shipping?,"Yes, orders over $50 qualify for free standard...",Shipping


In [28]:
# ============================================================
# Section 8: Dataset Shape
# ============================================================

print("="*60)
print("Dataset Shape")
print("="*60)

print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Dataset Shape
Rows : 500
Columns : 3


In [29]:
# ============================================================
# Section 9: Missing Values
# ============================================================

print("="*60)
print("Missing Values")
print("="*60)

print(df.isnull().sum())

Missing Values
question    0
answer      0
category    0
dtype: int64


In [30]:
# ============================================================
# Section 10: Duplicate Records
# ============================================================

duplicates = df.duplicated().sum()

print("="*60)
print("Duplicate Records")
print("="*60)

print(duplicates)

Duplicate Records
0


In [31]:
# ============================================================
# Section 11: Data Cleaning
# ============================================================

print("Before Cleaning :", df.shape)

df = df.dropna()

df = df.drop_duplicates()

df.reset_index(drop=True, inplace=True)

print("After Cleaning :", df.shape)

Before Cleaning : (500, 3)
After Cleaning : (500, 3)


In [32]:
# ============================================================
# Section 12: Categories
# ============================================================

print(df["category"].value_counts())

category
Orders                  143
Shipping                106
Payment                  74
Returns                  62
Product Info             37
Account                  33
Sizing                   15
Order Management         11
Support                   6
Student Records           2
IT Support                2
Faculty Support           1
Campus Services           1
IT Access                 1
Academic Information      1
Classroom Support         1
Alumni Support            1
Web Support               1
Accessibility             1
IT Security               1
Name: count, dtype: int64


In [33]:
# ============================================================
# Section 13: Text Cleaning
# ============================================================

import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"[^a-zA-Z ]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [34]:
# ============================================================
# Section 14: Apply Cleaning
# ============================================================

df["clean_question"] = df["question"].apply(clean_text)

df.head()

,question,answer,category,clean_question
0,Where is my order?,You can track your order using the tracking li...,Shipping,where is my order
1,How do I return a shirt that doesn't fit?,You can return any item within 30 days in its ...,Returns,how do i return a shirt that doesnt fit
2,"My package hasn't arrived yet, what should I do?",Please check the tracking number. If it hasn't...,Shipping,my package hasnt arrived yet what should i do
3,Can I exchange a dress for a different size?,"Yes, use our exchange form online to select th...",Returns,can i exchange a dress for a different size
4,Do you offer free shipping?,"Yes, orders over $50 qualify for free standard...",Shipping,do you offer free shipping


In [35]:
# ============================================================
# Section 15: TF-IDF Vectorization
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

faq_vectors = tfidf_vectorizer.fit_transform(df["clean_question"])

print("="*60)
print("TF-IDF Vectorization Completed")
print("="*60)

print("Vector Shape :", faq_vectors.shape)

TF-IDF Vectorization Completed
Vector Shape : (500, 334)


In [36]:
# ============================================================
# Section 16: Similarity Function
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

def get_best_answer(user_question):

    cleaned_question = clean_text(user_question)

    user_vector = tfidf_vectorizer.transform([cleaned_question])

    similarity = cosine_similarity(user_vector, faq_vectors)

    best_index = similarity.argmax()

    best_score = similarity[0][best_index]

    answer = df.iloc[best_index]["answer"]

    return answer, best_score

In [37]:
# ============================================================
# Section 17: Test Chatbot
# ============================================================

question = "How can I reset my password?"

answer, score = get_best_answer(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nSimilarity Score:")
print(round(score * 100, 2), "%")

Question:
How can I reset my password?

Answer:
Use the 'Forgot Password' link on the login page to reset your password.

Similarity Score:
100.0 %


In [38]:
# ============================================================
# Section 18: Save Model
# ============================================================

import joblib

joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

joblib.dump(faq_vectors, "faq_vectors.pkl")

joblib.dump(df, "faq_dataset.pkl")

print("="*60)
print("Model Files Saved Successfully")
print("="*60)

Model Files Saved Successfully


In [39]:
df.head()

,question,answer,category,clean_question
0,Where is my order?,You can track your order using the tracking li...,Shipping,where is my order
1,How do I return a shirt that doesn't fit?,You can return any item within 30 days in its ...,Returns,how do i return a shirt that doesnt fit
2,"My package hasn't arrived yet, what should I do?",Please check the tracking number. If it hasn't...,Shipping,my package hasnt arrived yet what should i do
3,Can I exchange a dress for a different size?,"Yes, use our exchange form online to select th...",Returns,can i exchange a dress for a different size
4,Do you offer free shipping?,"Yes, orders over $50 qualify for free standard...",Shipping,do you offer free shipping
